# 1. Setup and Security Configuration
## 1.1 Initialize Databricks Secrets
Setting up the secret scope to securely store Azure keys without hardcoding credentials.


In [0]:
# creation of the secret scope
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceAlreadyExists

wc = WorkspaceClient()
try:
    wc.secrets.create_scope(scope="retail_scope")
    print("Secret scope 'retail_scope' successfully created!")
except ResourceAlreadyExists:
    print("Secret scope 'retail_scope' already exists - skipping creation.")

Secret scope 'retail_scope' already exists - skipping creation.


In [0]:
# saving the key in the vault 
wc.secrets.put_secret(
    scope="retail_scope",
    key="adls_key",
    # Ive removed mine for security purposes
    string_value="api_string_pasted_here_"
)
print("Key securely saved in the vault!")

Key securely saved in the vault!


## 1.2 Configure Azure Storage Access
Establishing connection parameters to the Azure Data Lake Storage (ADLS Gen2) container.


In [0]:
STORAGE_ACCOUNT = 'datalakeretail'
SECRET_KEY = dbutils.secrets.get("retail_scope", "adls_key")

# function to create authenticated Azure paths for each folder 
def get_azure_path(container, path=""):
    """Generate an authenticated abfss:// path for Azure Data Lake"""
    base_path = f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
    if path:
        return f"{base_path}/{path}"
    return base_path

print("Azure Storage access configured!")

Azure Storage access configured!


# 2. Bronze Layer: Raw Data Ingestion
## 2.1 Define Container Paths
Generating secure `abfss://` paths for the raw data directories.


In [0]:
paths = ['customer', 'store', 'product', 'transaction']
container = 'retail-data-lake-container'
full_azure_path = {}

for path in paths:
    bronze_path = f"bronze/{path}"
    full_azure_path[path] = get_azure_path(container, bronze_path)
print(f"full azure paths saved seccussufully in :\n {full_azure_path}")


full azure paths saved seccussufully in :
 {'customer': 'abfss://retail-data-lake-container@datalakeretail.dfs.core.windows.net/bronze/customer', 'store': 'abfss://retail-data-lake-container@datalakeretail.dfs.core.windows.net/bronze/store', 'product': 'abfss://retail-data-lake-container@datalakeretail.dfs.core.windows.net/bronze/product', 'transaction': 'abfss://retail-data-lake-container@datalakeretail.dfs.core.windows.net/bronze/transaction'}


## 2.2 Load Bronze Data (Parquet)
Reading the raw Parquet datasets into PySpark DataFrames.


In [0]:
dataframes = {}
KEY_URL = f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net"
for dataset, azure_path_url in full_azure_path.items():
    dataframes[dataset] = (spark.read
                                .format("parquet")
                                .option(KEY_URL, SECRET_KEY)
                                .load(azure_path_url)
    )
print('All datasets loaded successfully')


All datasets loaded successfully


dfc : df customer  
dfs : df store  
dfp : df product  
dft : df transaction  

In [0]:
dfc = dataframes['customer']
dfs = dataframes['store']
dfp = dataframes['product']
dft = dataframes['transaction']

dfc.show(3)

customer_id,first_name,last_name,email,phone,city,registration_date
101,Youssef,Alaoui,user101@example.com,0612345678,Casablanca,2023-09-14
102,Salma,Bennani,user102@example.com,0623456789,Rabat,2024-01-21
103,Omar,El Idrissi,user103@example.com,0634567890,Marrakech,2023-07-10
104,Imane,Tazi,user104@example.com,0645678901,Fès,2024-02-05
105,Hamza,Amrani,user105@example.com,0656789012,Tanger,2023-06-28
106,Khadija,Lahlou,user106@example.com,0667890123,Agadir,2024-03-10
107,Mehdi,Chraibi,user107@example.com,0678901234,Meknès,2023-05-12
108,Nadia,Fassi,user108@example.com,0689012345,Oujda,2023-08-19
109,Ayoub,Berrada,user109@example.com,0690123456,Casablanca,2024-04-01
110,Sara,Mernissi,user110@example.com,0701234567,Rabat,2023-10-14


In [0]:
dft.show(3)

transaction_id,customer_id,product_id,store_id,quantity,transaction_date
1,227,8,4,4,2025-03-31
2,205,3,4,5,2024-11-12
3,216,2,2,3,2025-05-01
4,220,8,1,1,2024-11-02
5,205,5,2,1,2025-03-17
6,210,7,3,5,2025-01-04
7,210,7,2,5,2025-01-01
8,226,7,5,2,2025-06-08
9,223,1,3,2,2024-10-08
10,224,2,2,5,2024-08-27


In [0]:
dfs.show(3)

store_id,store_name,location
1,Marjane Mall,Casablanca
2,Carrefour Market,Rabat
3,ElectroPlanet,Marrakech
4,Aswak Assalam,Fès
5,Tachfine Center,Tanger


In [0]:
dfp.show(3)

product_id,product_name,category,price
1,Souris Sans Fil,Électronique,149.99
2,Enceinte Bluetooth,Électronique,399.50
3,Tapis de Sport,Fitness,199.00
4,Support PC Portable,Accessoires,249.99
5,Cahier Atlas,Papeterie,35.00
6,Bouteille Isotherme,Fitness,89.00
7,Montre Connectée,Électronique,1299.00
8,Organiseur de Bureau,Accessoires,99.00
9,Haltères,Fitness,549.00
10,Clé USB 32GB,Électronique,119.00


checking duplicated rows :

# 3. Data Quality & Profiling
## 3.1 Customer Duplicates Check
Identifying duplicate customer records before cleaning.


In [0]:
from pyspark.sql.functions import col, lit, concat, count
dfc.withColumn('full_name',concat(col('first_name'), 
                                  lit(' '), 
                                  col('last_name'))).groupBy('full_name', 'customer_id') \
                                                   .agg(count('*').alias('duplicated_times')) \
                                                   .filter(col('duplicated_times') > 1) \
                                                   .orderBy(col('duplicated_times'), ascending = False).show()

+---------+-----------+----------------+
|full_name|customer_id|duplicated_times|
+---------+-----------+----------------+
+---------+-----------+----------------+



checking if the customer_id is duplicated :

In [0]:
dfc.withColumn('full_name',concat(col('first_name'), 
                                  lit(' '), 
                                  col('last_name'))).groupBy('full_name', 'customer_id') \
                                                   .agg(count('customer_id').alias('id_duplicated_times')) \
                                                   .filter(col('id_duplicated_times') > 1) \
                                                   .orderBy(col('id_duplicated_times'), ascending = False).show()

+---------+-----------+-------------------+
|full_name|customer_id|id_duplicated_times|
+---------+-----------+-------------------+
+---------+-----------+-------------------+



# 4. Silver Layer: Data Cleaning and Standardization
## 4.1 Clean Customer Data
Standardizing strings, fixing phone numbers, and converting dates.


In [0]:
from pyspark.sql.functions import col, trim, lower, upper, regexp_replace, to_date, year, month, dayofmonth

dfc_clean = dfc \
    .withColumn("customer_id", col("customer_id").cast("int")) \
    .withColumn("first_name", trim(lower(col("first_name")))) \
    .withColumn("last_name", trim(upper(col("last_name")))) \
    .withColumn("email", trim(lower(col("email")))) \
    .withColumn("phone", regexp_replace(col("phone"), "[^0-9]", "")) \
    .withColumn("city", trim(upper(col("city")))) \
    .withColumn("registration_date", to_date(col("registration_date"), "yyyy-MM-dd"))
dfc_clean.show(10)

+-----------+----------+-----------+-------------------+----------+----------+-----------------+
|customer_id|first_name|  last_name|              email|     phone|      city|registration_date|
+-----------+----------+-----------+-------------------+----------+----------+-----------------+
|        101|   youssef|     ALAOUI|user101@example.com|0612345678|CASABLANCA|       2023-09-14|
|        102|     salma|    BENNANI|user102@example.com|0623456789|     RABAT|       2024-01-21|
|        103|      omar| EL IDRISSI|user103@example.com|0634567890| MARRAKECH|       2023-07-10|
|        104|     imane|       TAZI|user104@example.com|0645678901|       FÈS|       2024-02-05|
|        105|     hamza|     AMRANI|user105@example.com|0656789012|    TANGER|       2023-06-28|
|        106|   khadija|     LAHLOU|user106@example.com|0667890123|    AGADIR|       2024-03-10|
|        107|     mehdi|    CHRAIBI|user107@example.com|0678901234|    MEKNÈS|       2023-05-12|
|        108|     nadia|      

## 3.2 Product Duplicates Check
Profiling product data for anomalies and duplicate rows.


In [0]:
from pyspark.sql.functions import col, count
dfp.groupBy('product_name') \
            .agg(count('*').alias('total_rows')) \
            .filter(col('total_rows') > 1) \
            .orderBy(col('total_rows'), ascending = False) \
            .select( 'product_name', 'total_rows').show()

+------------+----------+
|product_name|total_rows|
+------------+----------+
+------------+----------+



## 4.2 Clean Product Data
Deduplication and price casting.


In [0]:
dfp_clean = dfp\
    .withColumn('product_id', col('product_id').cast('int')) \
    .dropDuplicates(['product_id']) \
    .withColumn('product_name', trim(lower(col('product_name')))) \
    .withColumn('category', trim(lower(col('category')))) \
    .withColumn('price', col('price').cast('decimal(10,2)')) \
    .orderBy(col('product_id'), ascending = True)
dfp_clean.show(10)

+----------+--------------------+------------+-------+
|product_id|        product_name|    category|  price|
+----------+--------------------+------------+-------+
|         1|     souris sans fil|électronique| 149.99|
|         2|  enceinte bluetooth|électronique| 399.50|
|         3|      tapis de sport|     fitness| 199.00|
|         4| support pc portable| accessoires| 249.99|
|         5|        cahier atlas|   papeterie|  35.00|
|         6| bouteille isotherme|     fitness|  89.00|
|         7|    montre connectée|électronique|1299.00|
|         8|organiseur de bureau| accessoires|  99.00|
|         9|            haltères|     fitness| 549.00|
|        10|        clé usb 32gb|électronique| 119.00|
+----------+--------------------+------------+-------+



## 4.3 Clean Store Data
Standardizing store locations and IDs.


In [0]:
dfs_clean = dfs \
    .withColumn("store_id", col("store_id").cast("int")) \
    .withColumn("store_name", trim(lower(col("store_name")))) \
    .withColumn("location", trim(upper(col('location'))))\
    .withColumnRenamed("location", 'store_location')\
    .orderBy(col('store_id'), ascending = True)

dfs_clean.show()

+--------+----------------+--------------+
|store_id|      store_name|store_location|
+--------+----------------+--------------+
|       1|    marjane mall|    CASABLANCA|
|       2|carrefour market|         RABAT|
|       3|   electroplanet|     MARRAKECH|
|       4|   aswak assalam|           FÈS|
|       5| tachfine center|        TANGER|
+--------+----------------+--------------+



In [0]:
dfs_clean = dfs \
    .withColumn("store_id", col("store_id").cast("int")) \
    .withColumn("store_name", trim(lower(col("store_name")))) \
    .withColumn("location", trim(upper(col('location'))))\
    .withColumnRenamed("location", 'store_location')\
    .orderBy(col('store_id'), ascending = True)

dfs_clean.show()

+--------+----------------+--------------+
|store_id|      store_name|store_location|
+--------+----------------+--------------+
|       1|    marjane mall|    CASABLANCA|
|       2|carrefour market|         RABAT|
|       3|   electroplanet|     MARRAKECH|
|       4|   aswak assalam|           FÈS|
|       5| tachfine center|        TANGER|
+--------+----------------+--------------+



## 4.4 Clean Transaction Data
Casting numeric types and formatting transaction dates.


In [0]:
dft_clean = dft \
    .withColumn('transaction_id', col("transaction_id").cast("int")) \
    .withColumn("customer_id", col("customer_id").cast("int")) \
    .withColumn("product_id", col("product_id").cast("int")) \
    .withColumn("store_id", col("store_id").cast("int")) \
    .withColumn('quantity', col("quantity").cast("int")) \
    .withColumn("transaction_date", to_date(col("transaction_date"), "yyyy-MM-dd"))
dft_clean.show()

+--------------+-----------+----------+--------+--------+----------------+
|transaction_id|customer_id|product_id|store_id|quantity|transaction_date|
+--------------+-----------+----------+--------+--------+----------------+
|             1|        227|         8|       4|       4|      2025-03-31|
|             2|        205|         3|       4|       5|      2024-11-12|
|             3|        216|         2|       2|       3|      2025-05-01|
|             4|        220|         8|       1|       1|      2024-11-02|
|             5|        205|         5|       2|       1|      2025-03-17|
|             6|        210|         7|       3|       5|      2025-01-04|
|             7|        210|         7|       2|       5|      2025-01-01|
|             8|        226|         7|       5|       2|      2025-06-08|
|             9|        223|         1|       3|       2|      2024-10-08|
|            10|        224|         2|       2|       5|      2024-08-27|
|            11|        2

## 4.5 Write to Silver Layer
Saving the cleansed DataFrames to ADLS in Delta format.


In [0]:
abbriviation_dict = {'c': 'customer', 's': 'store', 'p': 'product', 't': 'transaction'}
base_path = f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
KEY_URL = f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net"

for dict_key, dict_value in abbriviation_dict.items():
    sink_path = f"{base_path}/silver/{dict_value}/{dict_value}_slv.delta"
    globals()[f'df{dict_key}_clean'].write.mode('overwrite')\
                                           .format('delta')\
                                           .option(KEY_URL, SECRET_KEY)\
                                           .save(sink_path)
    print(f"{dict_value}_slv.parquet file has been saved seccussfully in this path\n     -->sink path :{sink_path}\n")                              
print('All files have been saved successfully!')

customer_slv.parquet file has been saved seccussfully in this path
     -->sink path :abfss://retail-data-lake-container@datalakeretail.dfs.core.windows.net/silver/customer/customer_slv.delta

store_slv.parquet file has been saved seccussfully in this path
     -->sink path :abfss://retail-data-lake-container@datalakeretail.dfs.core.windows.net/silver/store/store_slv.delta

product_slv.parquet file has been saved seccussfully in this path
     -->sink path :abfss://retail-data-lake-container@datalakeretail.dfs.core.windows.net/silver/product/product_slv.delta

transaction_slv.parquet file has been saved seccussfully in this path
     -->sink path :abfss://retail-data-lake-container@datalakeretail.dfs.core.windows.net/silver/transaction/transaction_slv.delta

All files have been saved successfully!


## 4.6 Reload Silver Data
Loading the cleaned Delta tables back into memory for the Gold transformation.


In [0]:
abbreviation_dict = {'c': 'customer', 's': 'store', 'p': 'product', 't': 'transaction'}
base_path = f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
KEY_URL = f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net"
silver_datasets = {}

for dict_key, dict_value in abbreviation_dict.items():
    source_path = f"{base_path}/silver/{dict_value}/{dict_value}_slv.delta"
    silver_datasets[dict_value] = spark.read.format('delta')\
                                            .option(KEY_URL, SECRET_KEY)\
                                            .load(source_path)
    globals()[f'df{dict_key*2}'] = silver_datasets[dict_value] 
    print(f"{dict_value}_slv.delta file has been loaded successfully")
print('All files have been loaded successfully!')
                                            

customer_slv.delta file has been loaded successfully
store_slv.delta file has been loaded successfully
product_slv.delta file has been loaded successfully
transaction_slv.delta file has been loaded successfully
All files have been loaded successfully!


In [0]:
dfcc.show(3)
dfpp.show(3)
dfss.show(3)
dftt.show(3)

+-----------+----------+----------+-------------------+----------+----------+-----------------+
|customer_id|first_name| last_name|              email|     phone|      city|registration_date|
+-----------+----------+----------+-------------------+----------+----------+-----------------+
|        101|   youssef|    ALAOUI|user101@example.com|0612345678|CASABLANCA|       2023-09-14|
|        102|     salma|   BENNANI|user102@example.com|0623456789|     RABAT|       2024-01-21|
|        103|      omar|EL IDRISSI|user103@example.com|0634567890| MARRAKECH|       2023-07-10|
+-----------+----------+----------+-------------------+----------+----------+-----------------+
only showing top 3 rows
+----------+------------------+------------+------+
|product_id|      product_name|    category| price|
+----------+------------------+------------+------+
|         1|   souris sans fil|électronique|149.99|
|         2|enceinte bluetooth|électronique|399.50|
|         3|    tapis de sport|     fitness|

# 5. Gold Layer: Star Schema and Aggregations
## 5.1 Create Dimension Tables
Building the `dim_store`, `dim_customer`, and `dim_product` tables.


In [0]:
from pyspark.sql.functions import col, concat, lit
df_dim_store = dfss\
        .withColumn('store_id', col('store_id'))\
        .withColumn('store_name', col('store_name'))\
        .withColumn('store_city', col('store_location'))\
        .withColumn('store_address', concat(col('store_name'),
                                            lit(', '),
                                            col('store_location'),
                                            lit(', Maroc')
                                            )
                    )

In [0]:
df_dim_customer = dfcc\
    .withColumn('customer_id', col('customer_id'))\
    .withColumn('customer_name', concat(col('first_name'), 
                                    lit(' '), 
                                    col('last_name')
                                    )
                )\
    .withColumn('first_name', col('first_name'))\
    .withColumn('last_name', col('last_name'))\
    .withColumn('customer_email', col('email'))\
    .withColumn('customer_phone', col('phone'))\
    .withColumn('customer_city', col('city'))\
    .withColumn('registration_date', col('registration_date'))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col
window_spec = Window.partitionBy('category')\
                    .orderBy(col('price').desc())

df_dim_product = dfpp\
    .withColumn('product_id', col('product_id'))\
    .withColumn('product_name', col('product_name'))\
    .withColumn('product_category', col('category'))\
    .withColumn('product_price', col('price'))\
    .withColumn('product_price_tier', rank().over(window_spec)
                )


## 5.2 Create Fact Table
Joining dimensions to create `fact_transaction` and calculating customer loyalty ranks.


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col, sum, lit, concat

window_customer = Window.partitionBy('customer_id')
window_rank = Window.orderBy(col('customer_total_quantity').desc())

                
df_fact_transaction = dftt\
    .join(dfpp, "product_id")\
    .join(dfcc, "customer_id")\
    .withColumn('total_amount', col("quantity")*col('price'))\
    .withColumn('customer_total_quantity', sum('quantity').over(window_customer))\
    .withColumn('loyal_customers_rank', dense_rank().over(window_rank))\
    .withColumn('customer_name', concat(col('first_name'),
                                        lit(' '),
                                        col('last_name'))
                )\
    .select('transaction_id', 
            'transaction_date',
            'customer_id', 
            'product_id', 
            'store_id', 
            'quantity', 
            'total_amount',
            'customer_name',
            'customer_total_quantity',
            'loyal_customers_rank'
            )\
    .orderBy(col('loyal_customers_rank'))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## 5.3 Compute Business KPIs
Generating the `general_kpis_table` to summarize revenue, store counts, and transaction averages.


In [0]:
from pyspark.sql.functions import countDistinct, sum, count, col
general_kpis_table = dftt\
    .join(dfpp, 'product_id')\
    .withColumn("line_total", col("quantity") * col("price"))\
    .groupBy()\
    .agg(
        count('transaction_id').alias('total_transactions'),
        countDistinct('customer_id').alias('total_customers'),
        countDistinct('product_id').alias('total_products'),
        countDistinct('store_id').alias('total_stores'),
        sum('line_total').alias('total_amount'),
        sum('quantity').alias('total_quantity_sold')
    )\
    .withColumn('avg_quantity_per_transaction', col('total_quantity_sold')/col('total_transactions'))\
    .withColumn('avg_amount_per_transaction', col('total_amount')/col('total_transactions'))\
    .select('total_transactions',
            'total_customers',
            'total_products',
            'total_stores',
            'total_amount',
            'total_quantity_sold',
            'avg_quantity_per_transaction',
            'avg_amount_per_transaction'
            )

print('General KPIs table has been created successfully')

General KPIs table has been created successfully


## 5.4 Compute Store Performance
Ranking the most sold product categories per store.


In [0]:
from pyspark.sql.functions import countDistinct, sum, count, col, dense_rank
from pyspark.sql.window import Window

window_store_product = Window.partitionBy('store_id', 'product_id')
window_rank = Window.partitionBy('store_id', 'product_category').orderBy(col('quantity').desc())

stores_performance = dftt\
    .join(dfpp, 'product_id')\
    .join(dfss, 'store_id')\
    .withColumnRenamed('category', 'product_category')\
    .withColumn('total_quantity_sold', sum('quantity').over(window_store_product))\
    .withColumn('total_amount_sold', sum(col('quantity')*col('price')).over(window_store_product))\
    .withColumn('most_sold_category', dense_rank().over(window_rank))\
    .select(
        'store_name',
        'product_category',
        'total_quantity_sold',
        'total_amount_sold',
        'most_sold_category')

    

# 6. Export and Visualization
## 6.1 Save Gold Tables (Delta)
Writing the final dimensional model back to Azure.


In [0]:
dfs_to_table_name = {'df_dim_customer' : 'dim_customer',
                     'df_dim_product' : 'dim_product',
                    'df_dim_store' : 'dim_store',
                    'df_fact_transaction' : 'fact_transaction',
                    'general_kpis_table' : 'general_kpis_table',
                     'stores_performance' : 'stores_performance'}

base_path = f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
KEY_URL = f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net"
i = 0
for df_name, table in dfs_to_table_name.items():
    sink_gold_path = f"{base_path}/gold/{table}.delta"

    df = globals()[df_name]
    df.write.mode('overwrite')\
      .format('delta')\
      .option(KEY_URL, SECRET_KEY)\
      .save(sink_gold_path)
    i+=1
    print(f'Table {table} has been created in gold layer successfully!')
print(f'all {i} tables have been created successfully in gold layer!')

Table dim_customer has been created in gold layer successfully!
Table dim_product has been created in gold layer successfully!
Table dim_store has been created in gold layer successfully!


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Table fact_transaction has been created in gold layer successfully!
Table general_kpis_table has been created in gold layer successfully!
Table stores_performance has been created in gold layer successfully!
all 6 tables have been created successfully in gold layer!


## 6.2 Export as CSV (Cloud)
Coalescing and saving flat CSVs to Azure for external consumption.


In [0]:
# Export DataFrames as CSV to Azure storage
base_path = f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
KEY_URL = f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net"

for df_name, table in dfs_to_table_name.items():
    csv_path = f"{base_path}/gold/csv/{table}.csv"
    
    # Write as single CSV file with header
    globals()[df_name].coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .option(KEY_URL, SECRET_KEY) \
        .csv(csv_path)
    
    print(f"Saved {table}.csv to Azure storage: {csv_path}")

print("All tables exported to Azure storage as CSV files!")

Saved dim_customer.csv to Azure storage: abfss://retail-data-lake-container@datalakeretail.dfs.core.windows.net/gold/csv/dim_customer.csv
Saved dim_product.csv to Azure storage: abfss://retail-data-lake-container@datalakeretail.dfs.core.windows.net/gold/csv/dim_product.csv
Saved dim_store.csv to Azure storage: abfss://retail-data-lake-container@datalakeretail.dfs.core.windows.net/gold/csv/dim_store.csv


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Saved fact_transaction.csv to Azure storage: abfss://retail-data-lake-container@datalakeretail.dfs.core.windows.net/gold/csv/fact_transaction.csv
Saved general_kpis_table.csv to Azure storage: abfss://retail-data-lake-container@datalakeretail.dfs.core.windows.net/gold/csv/general_kpis_table.csv
Saved stores_performance.csv to Azure storage: abfss://retail-data-lake-container@datalakeretail.dfs.core.windows.net/gold/csv/stores_performance.csv
All tables exported to Azure storage as CSV files!


## 6.3 Export Locally
Saving Pandas representations directly to the local filesystem.


In [0]:
import os

# Create folder
folder = "C:/Users/rachid/3D Objects/gold_tables_retail_data"
os.makedirs(folder, exist_ok=True)

for df_name, table in dfs_to_table_name.items():
    # Save directly as single CSV
    globals()[df_name].toPandas().to_csv(f"{folder}/{table}.csv", index=False)
    print(f"Saved {table}.csv")

print("All tables saved!")

Saved dim_customer.csv
Saved dim_product.csv
Saved dim_store.csv


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Saved fact_transaction.csv
Saved general_kpis_table.csv
Saved stores_performance.csv
All tables saved!


## 6.4 Visual Verification
Displaying the final Gold tables.


In [0]:
# Display each table - click the download icon (⬇) in the table header to save to your laptop

print("=== DIM_CUSTOMER ===")
display(df_dim_customer)

print("\n=== DIM_PRODUCT ===")
display(df_dim_product)

print("\n=== DIM_STORE ===")
display(df_dim_store)

print("\n=== FACT_TRANSACTION ===")
display(df_fact_transaction)

print("\n=== GENERAL_KPIS_TABLE ===")
display(general_kpis_table)

print("\n=== STORES_PERFORMANCE ===")
display(stores_performance)

=== DIM_CUSTOMER ===


customer_id,first_name,last_name,email,phone,city,registration_date,customer_name,customer_email,customer_phone,customer_city
101,youssef,ALAOUI,user101@example.com,0612345678,CASABLANCA,2023-09-14,youssef ALAOUI,user101@example.com,0612345678,CASABLANCA
102,salma,BENNANI,user102@example.com,0623456789,RABAT,2024-01-21,salma BENNANI,user102@example.com,0623456789,RABAT
103,omar,EL IDRISSI,user103@example.com,0634567890,MARRAKECH,2023-07-10,omar EL IDRISSI,user103@example.com,0634567890,MARRAKECH
104,imane,TAZI,user104@example.com,0645678901,FÈS,2024-02-05,imane TAZI,user104@example.com,0645678901,FÈS
105,hamza,AMRANI,user105@example.com,0656789012,TANGER,2023-06-28,hamza AMRANI,user105@example.com,0656789012,TANGER
106,khadija,LAHLOU,user106@example.com,0667890123,AGADIR,2024-03-10,khadija LAHLOU,user106@example.com,0667890123,AGADIR
107,mehdi,CHRAIBI,user107@example.com,0678901234,MEKNÈS,2023-05-12,mehdi CHRAIBI,user107@example.com,0678901234,MEKNÈS
108,nadia,FASSI,user108@example.com,0689012345,OUJDA,2023-08-19,nadia FASSI,user108@example.com,0689012345,OUJDA
109,ayoub,BERRADA,user109@example.com,0690123456,CASABLANCA,2024-04-01,ayoub BERRADA,user109@example.com,0690123456,CASABLANCA
110,sara,MERNISSI,user110@example.com,0701234567,RABAT,2023-10-14,sara MERNISSI,user110@example.com,0701234567,RABAT



=== DIM_PRODUCT ===


product_id,product_name,category,price,product_category,product_price,product_price_tier
4,support pc portable,accessoires,249.99,accessoires,249.99,1
8,organiseur de bureau,accessoires,99.00,accessoires,99.00,2
9,haltères,fitness,549.00,fitness,549.00,1
3,tapis de sport,fitness,199.00,fitness,199.00,2
6,bouteille isotherme,fitness,89.00,fitness,89.00,3
5,cahier atlas,papeterie,35.00,papeterie,35.00,1
7,montre connectée,électronique,1299.00,électronique,1299.00,1
2,enceinte bluetooth,électronique,399.50,électronique,399.50,2
1,souris sans fil,électronique,149.99,électronique,149.99,3
10,clé usb 32gb,électronique,119.00,électronique,119.00,4



=== DIM_STORE ===


...

transaction_id,transaction_date,customer_id,product_id,store_id,quantity,total_amount,customer_name,customer_total_quantity,loyal_customers_rank



=== GENERAL_KPIS_TABLE ===


total_transactions,total_customers,total_products,total_stores,total_amount,total_quantity_sold,avg_quantity_per_transaction,avg_amount_per_transaction
30,21,8,5,35262.82,101,3.3666666666666667,1175.427333333



=== STORES_PERFORMANCE ===


store_name,product_category,total_quantity_sold,total_amount_sold,most_sold_category
marjane mall,accessoires,2,198.00,1
marjane mall,accessoires,2,198.00,1
marjane mall,papeterie,6,210.00,1
marjane mall,papeterie,6,210.00,2
carrefour market,fitness,6,534.00,1
carrefour market,fitness,6,534.00,2
carrefour market,papeterie,1,35.00,1
carrefour market,électronique,8,3196.00,1
carrefour market,électronique,5,6495.00,1
carrefour market,électronique,8,3196.00,2
